# Run dbt for Fabric Lakehouse

This notebook installs `dbt-fabricspark`, generates a local dbt profile for Fabric Notebook authentication, and runs the dbt project in this repository.

Update the workspace and lakehouse variables in Cell 2 before running the notebook.

In [ ]:
workspace_id = "<fabric-workspace-guid>"
lakehouse_id = "<fabric-lakehouse-guid>"
lakehouse_name = "bronze_lakehouse"
schema_name = "bronze_lakehouse"
dbt_project_dir = "/lakehouse/default/Files/FabricTestErran/src/Fabric/transform/dbt_fabric_project"

In [ ]:
%pip install -U dbt-fabricspark

In [ ]:
import os
from pathlib import Path

dbt_dir = Path.home() / '.dbt'
dbt_dir.mkdir(parents=True, exist_ok=True)
profiles_yml = f"""
fabric_etl_project:
  target: dev
  outputs:
    dev:
      type: fabricspark
      method: livy
      endpoint: https://api.fabric.microsoft.com/v1
      workspaceid: {workspace_id}
      lakehouseid: {lakehouse_id}
      lakehouse: {lakehouse_name}
      schema: {schema_name}
      authentication: fabric_notebook
      threads: 1
      retry_all: true
      reuse_session: true
      spark_config:
        name: fabric-dbt-notebook
        conf:
          spark.sql.caseSensitive: "false"
"""
(dbt_dir / 'profiles.yml').write_text(profiles_yml, encoding='utf-8')
print((dbt_dir / 'profiles.yml').read_text(encoding='utf-8'))

In [ ]:
%cd {dbt_project_dir}
!dbt debug
!dbt deps
!dbt build